# Solutions: JSON and CSV

These are the reference solutions for the Module 02 — JSON and CSV exercises.
Try to solve the exercises yourself first before checking here!

Data files used:
- `../../../data/synthetic/model_outputs.json`
- `../../../data/synthetic/evaluation_results.csv`

In [ ]:
import json
import csv
import pandas as pd
from pathlib import Path

JSON_PATH = Path("../../../data/synthetic/model_outputs.json")
CSV_PATH  = Path("../../../data/synthetic/evaluation_results.csv")

---
## Exercise 1 Solution: Load JSON and Count Records

**Task:** Load `model_outputs.json` and print how many records it contains.

In [ ]:
with open(JSON_PATH, "r") as f:
    data = json.load(f)

# json.load() parses the JSON into native Python objects:
# JSON object  -> Python dict
# JSON array   -> Python list
# JSON string  -> Python str
# JSON number  -> Python int or float

print(f"Type of loaded data: {type(data)}")
print(f"Number of records: {len(data)}")

# Peek at the first record to understand the structure
print("\nFirst record:")
print(json.dumps(data[0], indent=2))

---
## Exercise 2 Solution: Find Flagged IDs

**Task:** From `model_outputs.json`, collect the IDs of all records where `"flagged"` is `true`.

In [ ]:
with open(JSON_PATH, "r") as f:
    data = json.load(f)

# Method 1: for loop
flagged_ids = []
for record in data:
    if record.get("flagged") is True:
        flagged_ids.append(record["id"])

print(f"Flagged IDs (loop): {flagged_ids}")
print(f"Count: {len(flagged_ids)}")

# Method 2: list comprehension (more Pythonic)
flagged_ids_v2 = [r["id"] for r in data if r.get("flagged") is True]
print(f"\nFlagged IDs (comprehension): {flagged_ids_v2}")

# Note: use .get("flagged") rather than ["flagged"] to avoid KeyError
# if some records don't have that field.

---
## Exercise 3 Solution: Compute Average Score from CSV

**Task:** Load `evaluation_results.csv` using the `csv` module and compute the average score.

In [ ]:
scores = []

with open(CSV_PATH, "r", newline="") as f:
    # DictReader gives each row as a dict keyed by column headers.
    # Much more readable than plain reader which gives lists.
    reader = csv.DictReader(f)
    for row in reader:
        # CSV values are always strings — must convert to float!
        score = float(row["score"])
        scores.append(score)

average = sum(scores) / len(scores)
print(f"Number of rows: {len(scores)}")
print(f"Average score: {average:.4f}")
print(f"Min score: {min(scores):.4f}")
print(f"Max score: {max(scores):.4f}")

---
## Exercise 4 Solution: Pandas GroupBy — Per-Model Averages

**Task:** Use pandas to load `evaluation_results.csv` and compute the average score per model.

In [ ]:
# pandas makes this almost trivially easy compared to the csv module
df = pd.read_csv(CSV_PATH)

print("DataFrame shape:", df.shape)
print("\nColumn names:", df.columns.tolist())
print("\nFirst 3 rows:")
print(df.head(3))

# groupby('model') groups rows by the 'model' column,
# then ['score'].mean() computes the average score for each group.
per_model_avg = df.groupby("model")["score"].mean()
print("\nPer-model average scores:")
print(per_model_avg)

# More detailed stats per model
print("\nDetailed stats per model:")
print(df.groupby("model")["score"].agg(["mean", "std", "min", "max", "count"]))

---
## Key Takeaways

- `json.load(f)` parses a JSON file into Python objects; `json.dumps(obj, indent=2)` pretty-prints back to a string
- Use `.get("key")` instead of `["key"]` when the key might be missing — returns `None` instead of raising `KeyError`
- `csv.DictReader` gives you rows as dicts — much easier than indexing by position
- CSV values are always strings — convert to `int()` or `float()` before doing math
- `pd.read_csv()` + `groupby()` is the go-to pattern for computing per-group statistics in research